# IndustryGPT: Fine-Tuning a Pre-Trained LLM on History NCERT Data

This Jupyter Notebook contains the step-by-step workflow for fine-tuning a pre-trained Large Language Model (LLM) on NCERT History question-answering data. 

### Project Objective
- **Domain**: History NCERT (Indian History: Ancient, Medieval, Modern)
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct` or `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (Hugging Face)
- **Technique**: Parameter-Efficient Fine-Tuning (PEFT) using **QLoRA** (4-bit quantization + Low-Rank Adaptation) to enable training on a free Google Colab T4 GPU.
- **Max Epochs**: Limited to a maximum of 25 epochs (configured for 5-10 epochs for efficient training).

### How to run on Google Colab:
1. Open Google Colab and set the runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Upload this notebook and the dataset file `data/ncert_qa_dataset.json` (generated from `data_pipeline.py`).
3. Run the cells sequentially.

## Step 1: Install Required Libraries
We need to install Hugging Face's libraries for loading, quantizing, and fine-tuning models.

In [ ]:
# Install transformers, accelerate, peft, bitsandbytes (for quantization), and trl (for SFTTrainer)
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets

## Step 2: Import Libraries & Check GPU
We check if PyTorch can access the T4 GPU.

In [ ]:
import torch
import json
import pandas as pd
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## Step 3: Load and Preprocess the Dataset
We load `ncert_qa_dataset.json`, inspect it, and convert it into a Hugging Face `Dataset` with standard instruction-following formatting.

In [ ]:
# Load JSON dataset (upload ncert_qa_dataset.json to Colab directory first)
dataset_path = "ncert_qa_dataset.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} training examples from NCERT QA dataset.")
df = pd.DataFrame(data)
print(df.head())

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
print(dataset)

## Step 4: Formatting the Prompt (Instruction-Tuning Format)
We format our QA pairs into a chat template or prompt structure. For Qwen/TinyLlama, we use standard system/user/assistant roles.

In [ ]:
def format_prompts(batch):
    formatted_texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        # Construct prompt
        system_prompt = "You are an expert history tutor trained on Indian History NCERT textbooks. Answer the user's question accurately based on NCERT guidelines."
        user_content = inst + (f"\nInput: {inp}" if inp else "")
        
        # We format it using a standard chat-like message format
        text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

# Map the formatting function
dataset = dataset.map(format_prompts, batched=True)
print("Sample formatted text:")
print(dataset[0]["text"])

## Step 5: Load Pre-trained Base Model in 4-bit Quantization
To prevent Out-Of-Memory (OOM) errors, we quantize the model weights to 4-bits. We select **`Qwen/Qwen2.5-1.5B-Instruct`** as it is state-of-the-art for its size and supports multi-lingual and precise factual retrieval.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Set up 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully in 4-bit precision!")

## Step 6: Setup PEFT (LoRA)
We inject trainable Low-Rank Adaptation (LoRA) adapters into the model layers, significantly reducing parameters to train.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Define LoRA Config
peft_config = LoraConfig(
    r=16,                         # Rank
    lora_alpha=32,                # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target linear layers
    lora_dropout=0.05,            # Dropout probability
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

## Step 7: Define Training Configuration & Arguments
We configure training parameters. The prompt limits training to a maximum of 25 epochs. We set it to 10 epochs which is optimal to prevent overfitting on our Q&A dataset while ensuring convergence.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results_ncert",
    num_train_epochs=10,                 # Train for 10 epochs (under the 25 max limit)
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=5,
    learning_rate=2e-4,
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    report_to="none"
)

# Set up Supervised Fine-Tuning (SFT) Trainer
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Trainer initialized!")

## Step 8: Run Fine-Tuning
We start the training loop.

In [ ]:
print("Starting model fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

## Step 9: Save the Fine-Tuned Weights
We save the LoRA adapter weights and tokenizer to a local directory.

In [ ]:
output_dir = "./ncert_history_lora_adapter"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned LoRA adapters saved to {output_dir}")

## Step 10: Test the Fine-Tuned Model
Let's test the model on a question to verify that it generates historical facts correctly.

In [ ]:
from peft import PeftModel

# Load baseline model (already loaded in variable `model`)
# Load LoRA weights
test_model = PeftModel.from_pretrained(model, output_dir)
test_model.eval()

# Write a query
query = "Explain the features of the town planning and drainage system in the Harappan Civilisation."
prompt = f"<|im_start|>system\nYou are an expert history tutor trained on Indian History NCERT textbooks. Answer the user's question accurately based on NCERT guidelines.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("\n--- Query ---\n", query)
print("\n--- Generated Answer ---\n", response)